In [1]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import regex
import dtale
import quantstats as qs
import webbrowser
import sqlalchemy
from sqlalchemy import text
import yaml
import traceback
from pathlib import Path
from backtest_reporter_impl.done_processor import DoneProcessor

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 1024)
pd.set_option('display.width', None)

done_fname='20250925_0000_20251012_0000_done_20251023_2250.csv'
return_source='h'
initial_capital=1000000
ignore_volume=False
depth1_dir = r'D:\DataRepository\depth1'
bt_dir = r'D:\DataRepository\backtest'
market = 'okx'
instrument = 'BTC-USDT'
agg_interval = '1min'
dtale_host = 'localhost'
last_dtale_port = 40003

processor = DoneProcessor(done_fname='20250925_0000_20251012_0000_done_20251023_2250.csv', return_source='h', initial_capital=1000000, ignore_volume=False)

In [2]:

price_df = processor.get_return_data()
price_df['close_cum_ret'] = price_df['close'] / price_df['close'].iloc[0] - 1
price_df

使用小时K线收盘价作为收益率计算基准
sql=
        SELECT ex_time, close
        FROM kline_1H
        WHERE market = 'okx'
        AND instrument = 'BTC-USDT'
        AND ex_time >= 1758758400000000000
        AND ex_time <= 1760313600000000000
        ORDER BY ex_time
        
从MySQL加载了 408 条1HK线close数据


,close,close_cum_ret
ex_time,,
2025-09-25 00:00:00,113051.1,0.000000
2025-09-25 01:00:00,112673.0,-0.003345
2025-09-25 02:00:00,112901.4,-0.001324
2025-09-25 03:00:00,112541.5,-0.004508
2025-09-25 04:00:00,111740.1,-0.011597
...,...,...
2025-10-11 19:00:00,111184.3,-0.016513
2025-10-11 20:00:00,110259.0,-0.024698
2025-10-11 21:00:00,110900.1,-0.019027


In [3]:
fpath = os.path.join(processor.bt_dir, done_fname)
done_df = pd.read_csv(fpath)
done_df

,policy_no,private_no,entno,market,account_id,inst_id,bs_side,pos_side,done_px,done_amt,filled_time
0,strat_fut_arbi_v3_BTC_USDT,29c9c735-1bcb-4e0f-8d4e-29e3bcb53278,1761259849000000001,okx,okx_account3_dummy,BTC-USDT-260327,s,NaN,117530.8,0.0001,1758758457240000000
1,strat_fut_arbi_v3_BTC_USDT,9c07eafe-d109-4023-9516-840242480b99,1761259849000000002,okx,okx_account3_dummy,BTC-USDT-251031,b,NaN,113897.5,0.0001,1758758457240000000
2,strat_fut_arbi_v3_BTC_USDT,65f10f13-0536-4a93-a9c6-91d38aeb4f51,1761259849000000003,okx,okx_account3_dummy,BTC-USDT-260327,b,NaN,123014.9,0.0001,1759449614102000000
3,strat_fut_arbi_v3_BTC_USDT,3ca487de-dec7-4c5a-a6f5-0eb56e603b2c,1761259849000000004,okx,okx_account3_dummy,BTC-USDT-251031,s,NaN,121161.9,0.0001,1759449614102000000
4,strat_fut_arbi_v3_BTC_USDT,e658783c-54a0-4ec0-9eed-37f6b925fb26,1761259849000000005,okx,okx_account3_dummy,BTC-USDT-260327,s,NaN,123014.8,0.0001,1760110481166000000
5,strat_fut_arbi_v3_BTC_USDT,25f7476b-b841-4927-a84c-48532594ad70,1761259849000000006,okx,okx_account3_dummy,BTC-USDT-251031,b,NaN,119008.7,0.0001,1760110481166000000
6,strat_fut_arbi_v3_BTC_USDT,fced1125-4d23-4970-ae10-598b32085574,1761259849000000007,okx,okx_account3_dummy,BTC-USDT-260327,b,NaN,123014.9,0.0001,1760110526156000000
7,strat_fut_arbi_v3_BTC_USDT,4399b921-1312-4d23-b766-c846cd17aba7,1761259849000000008,okx,okx_account3_dummy,BTC-USDT-251031,s,NaN,119361.4,0.0001,1760110526156000000
8,strat_fut_arbi_v3_BTC_USDT,2d508135-36eb-450d-b211-b820c90709ae,1761259849000000009,okx,okx_account3_dummy,BTC-USDT-260327,s,NaN,123014.8,0.0001,1760115003447000000
9,strat_fut_arbi_v3_BTC_USDT,f11b63f0-9bce-4a51-aa1c-464360832dbe,1761259849000000010,okx,okx_account3_dummy,BTC-USDT-251031,b,NaN,119004.8,0.0001,1760115003447000000


In [4]:
done_df = done_df.loc[:, ['inst_id', 'bs_side', 'pos_side', 'done_px', 'done_amt', 'filled_time']]
done_df["filled_time"] = pd.to_datetime(done_df["filled_time"], unit="ns")
done_df["agg_time"] = done_df["filled_time"].dt.floor(agg_interval)
done_df

# done_df = processor.process_done(processor.done_fname)
# done_df

,inst_id,bs_side,pos_side,done_px,done_amt,filled_time,agg_time
0,BTC-USDT-260327,s,NaN,117530.8,0.0001,2025-09-25 00:00:57.240,2025-09-25 00:00:00
1,BTC-USDT-251031,b,NaN,113897.5,0.0001,2025-09-25 00:00:57.240,2025-09-25 00:00:00
2,BTC-USDT-260327,b,NaN,123014.9,0.0001,2025-10-03 00:00:14.102,2025-10-03 00:00:00
3,BTC-USDT-251031,s,NaN,121161.9,0.0001,2025-10-03 00:00:14.102,2025-10-03 00:00:00
4,BTC-USDT-260327,s,NaN,123014.8,0.0001,2025-10-10 15:34:41.166,2025-10-10 15:34:00
5,BTC-USDT-251031,b,NaN,119008.7,0.0001,2025-10-10 15:34:41.166,2025-10-10 15:34:00
6,BTC-USDT-260327,b,NaN,123014.9,0.0001,2025-10-10 15:35:26.156,2025-10-10 15:35:00
7,BTC-USDT-251031,s,NaN,119361.4,0.0001,2025-10-10 15:35:26.156,2025-10-10 15:35:00
8,BTC-USDT-260327,s,NaN,123014.8,0.0001,2025-10-10 16:50:03.447,2025-10-10 16:50:00
9,BTC-USDT-251031,b,NaN,119004.8,0.0001,2025-10-10 16:50:03.447,2025-10-10 16:50:00


In [5]:
done_df["pos_change"] = done_df.apply(processor.calculate_position, axis=1)
done_df["cap_change"] = done_df['pos_change'] * -done_df['done_px']
done_df["pos_cumsum"] = done_df['pos_change'].cumsum()
done_df["cap_cumsum"] = done_df['cap_change'].cumsum()
done_df

,inst_id,bs_side,pos_side,done_px,done_amt,filled_time,agg_time,pos_change,cap_change,pos_cumsum,cap_cumsum
0,BTC-USDT-260327,s,NaN,117530.8,0.0001,2025-09-25 00:00:57.240,2025-09-25 00:00:00,-0.0001,11.75308,-0.0001,11.75308
1,BTC-USDT-251031,b,NaN,113897.5,0.0001,2025-09-25 00:00:57.240,2025-09-25 00:00:00,0.0001,-11.38975,0.0000,0.36333
2,BTC-USDT-260327,b,NaN,123014.9,0.0001,2025-10-03 00:00:14.102,2025-10-03 00:00:00,0.0001,-12.30149,0.0001,-11.93816
3,BTC-USDT-251031,s,NaN,121161.9,0.0001,2025-10-03 00:00:14.102,2025-10-03 00:00:00,-0.0001,12.11619,0.0000,0.17803
4,BTC-USDT-260327,s,NaN,123014.8,0.0001,2025-10-10 15:34:41.166,2025-10-10 15:34:00,-0.0001,12.30148,-0.0001,12.47951
5,BTC-USDT-251031,b,NaN,119008.7,0.0001,2025-10-10 15:34:41.166,2025-10-10 15:34:00,0.0001,-11.90087,0.0000,0.57864
6,BTC-USDT-260327,b,NaN,123014.9,0.0001,2025-10-10 15:35:26.156,2025-10-10 15:35:00,0.0001,-12.30149,0.0001,-11.72285
7,BTC-USDT-251031,s,NaN,119361.4,0.0001,2025-10-10 15:35:26.156,2025-10-10 15:35:00,-0.0001,11.93614,0.0000,0.21329
8,BTC-USDT-260327,s,NaN,123014.8,0.0001,2025-10-10 16:50:03.447,2025-10-10 16:50:00,-0.0001,12.30148,-0.0001,12.51477
9,BTC-USDT-251031,b,NaN,119004.8,0.0001,2025-10-10 16:50:03.447,2025-10-10 16:50:00,0.0001,-11.90048,0.0000,0.61429


In [6]:
sum_df = done_df.groupby('agg_time', as_index=False).agg(
    pos_change=('pos_change', 'sum'),  # 每小时仓位变动总和
    cap_change=('cap_change', 'sum')  # 每小时资金变动总和
)
# sum_df['bs_side'] = np.where(sum_df['pos_change'] >= 0, 'b', 's')
sum_df['buy'] = np.where(sum_df['pos_change'] > 0, sum_df['cap_change'] / sum_df['pos_change'], np.nan)
sum_df['sell'] = np.where(sum_df['pos_change'] < 0, sum_df['cap_change'] / sum_df['pos_change'], np.nan)

sum_df.set_index('agg_time', inplace=True, drop=True)
sum_df

,pos_change,cap_change,buy,sell
agg_time,,,,
2025-09-25 00:00:00,0.0,0.36333,NaN,NaN
2025-10-03 00:00:00,0.0,-0.18530,NaN,NaN
2025-10-10 15:34:00,0.0,0.40061,NaN,NaN
2025-10-10 15:35:00,0.0,-0.36535,NaN,NaN
2025-10-10 16:50:00,0.0,0.40100,NaN,NaN
